# Before/after robustness: one exposure per measurement, dam-level averages

`analysis_eda.ipynb` pools rows, so its averages can be driven by heavily sampled stations, by different dams contributing to different years or distances (composition bias), and by changes in how water was measured. This notebook re-asks the before/after question in a way that avoids those problems:

1. **One exposure per measurement.** Downstream rows only where the measurement is assigned to this dam (`exposure_status` `assigned_sole` / `assigned_nearest`: no competing removal). Upstream rows only as clean controls (`upstream_control`, not possibly in the former reservoir).
2. **Dam-level averages.** One change per dam-station pair (daily values, seasonal cycle removed), then one per dam, then an equal-weight average across dams with a dam-bootstrap 95% interval. Every summary reports how many dams it rests on.
3. **Matched control.** For dams with both, the downstream change minus the upstream change at the *same* dam (BACI), which removes each dam's regional trend.
4. **Measurement consistency.** The same estimate restricted to pairs measured by one provider, one reported unit (turbidity NTU / FNU) or one method throughout, and without censored values; plus detection limits across the removal.
5. **Event study by dam.** Each dam's own trajectory, the equal-weight mean, and a balanced panel (dams with data in every year shown), with the dams behind each point.
6. **Distance within dams.** Near vs far stations of the same dam, so differences between watersheds can't pose as a distance effect.

Reads `analysis_measurement.parquet` built on or after 2026-10-26 (link distance limits, `exposure_status`, `upstream_in_former_reservoir`, `result_unit_raw`, `method_id`). Nothing is written to the warehouse.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from etl.common import wq_rules
from etl.common.config import load_config
from etl.common.warehouse import warehouse_dir

DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
cfg = load_config(CASE_STUDY, project_root=DATA_ROOT)
WAREHOUSE = warehouse_dir(cfg)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

# ---- Settings ----
SERIES_OF_INTEREST = ["water_temp", "dissolved_oxygen", "ph", "specific_conductance", "turbidity",
                      "phosphorus (Total)", "orthophosphate (Dissolved)", "tds"]
MIN_DAMS = 5                       # summaries resting on fewer dams aren't estimated
N_BOOT = 1000                      # bootstrap resamples of dams
BALANCED_YEARS = [-2, -1, 1, 2]    # a dam is in the balanced panel if it has data in all of these
SEED = 0
LOG_PARAMETERS = {"turbidity", "specific_conductance", "nitrate", "ammonia",
                  "orthophosphate", "phosphorus", "tds"}
rng = np.random.default_rng(SEED)


def y_label(series):
    parameter = series.split(" (")[0]
    unit = wq_rules.STANDARD_UNITS.get(parameter, "")
    return f"log10 {unit}" if parameter in LOG_PARAMETERS else unit


def facet_axes(n, ncols=4, width=4, height=3):
    """A grid of n axes, extra ones hidden."""
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, height * nrows), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, list(axes.flat)


def dam_bootstrap(values):
    """Equal-weight mean of one value per dam, with a 95% CI from resampling dams."""
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) < MIN_DAMS:
        return np.nan, np.nan, np.nan
    idx = rng.integers(0, len(values), size=(N_BOOT, len(values)))
    low, high = np.percentile(values[idx].mean(axis=1), [2.5, 97.5])
    return values.mean(), low, high

## 1. One exposure per measurement

Each row gets a **role**:
- `treated`: downstream, and this dam is the measurement's assigned removal (no other removal upstream within the window, or the next one is clearly farther).
- `control`: upstream of this dam, not downstream of another removal in its window, and not possibly in the former reservoir (which the removal itself changes).
- `excluded`: everything else (competing or unresolved exposure, exposed or in-reservoir upstream stations).

The tables show what each rule removes, in rows and in dams.

In [ ]:
COLUMNS = ["measurement_id", "AR_ID", "station_id", "parameter_std", "sample_fraction_std", "side",
           "network_distance_km", "period", "years_from_removal", "sample_date", "month",
           "result_value_std", "is_censored", "provider", "result_unit_raw", "method_id",
           "exposure_status", "upstream_in_former_reservoir"]
df = pd.read_parquet(WAREHOUSE / "analysis_measurement.parquet", columns=COLUMNS)

df["series"] = df["parameter_std"].where(
    df["sample_fraction_std"].isna(),
    df["parameter_std"] + " (" + df["sample_fraction_std"].fillna("") + ")",
)
df = df[df["series"].isin(SERIES_OF_INTEREST)].copy()
df["y"] = np.where(df["parameter_std"].isin(LOG_PARAMETERS),
                   np.log10(df["result_value_std"].clip(lower=1e-3)), df["result_value_std"])

in_reservoir = df["upstream_in_former_reservoir"].fillna(False).astype(bool)
df["role"] = np.select(
    [(df["side"] == "downstream") & df["exposure_status"].isin(["assigned_sole", "assigned_nearest"]),
     (df["side"] == "upstream") & (df["exposure_status"] == "upstream_control") & ~in_reservoir],
    ["treated", "control"], "excluded")

summary = (df.assign(in_former_reservoir=in_reservoir)
           .groupby(["side", "exposure_status", "in_former_reservoir", "role"], dropna=False)
           .agg(rows=("measurement_id", "size"), dams=("AR_ID", "nunique"),
                stations=("station_id", "nunique")))
display(summary)
print(f"Kept: {(df['role'] != 'excluded').mean():.0%} of rows; "
      f"treated dams: {df.loc[df['role'] == 'treated', 'AR_ID'].nunique()}, "
      f"dams with a control: {df.loc[df['role'] == 'control', 'AR_ID'].nunique()}")
df = df[df["role"] != "excluded"]

## 2. Change per dam, and the matched (BACI) contrast

`pair_changes()` turns rows into one change per dam-station pair and series:
1. one median per station, series and day (a continuous logger doesn't outweigh grab samples);
2. anomaly = that value minus the station's mean for the series and calendar month (removes level and seasonal cycle);
3. change = mean anomaly after removal − mean anomaly before.

Pairs are then averaged within each dam, so a dam with ten stations counts once. Units are those of y (log10 for skewed parameters: 0.05 ≈ +12%). Three summaries per series:
- **treated**: the mean downstream change across dams, and the share of dams that went up;
- **control**: the same upstream (what changed regionally without the removal);
- **BACI**: downstream − upstream change at the same dam, for dams that have both. This is the main estimate.

In [ ]:
def pair_changes(frame):
    """One row per dam, station, role and series: mean seasonal anomaly after - before."""
    measurements = frame.drop_duplicates("measurement_id")
    daily = (measurements.groupby(["station_id", "series", "sample_date"], as_index=False)
             .agg(y=("y", "median"), month=("month", "first")))
    daily["anomaly"] = daily["y"] - daily.groupby(["station_id", "series", "month"])["y"].transform("mean")
    days = (frame[["AR_ID", "station_id", "role", "series", "sample_date", "period", "years_from_removal",
                   "network_distance_km"]].drop_duplicates(["AR_ID", "station_id", "series", "sample_date"])
            .merge(daily[["station_id", "series", "sample_date", "anomaly"]],
                   on=["station_id", "series", "sample_date"]))
    change = (days.groupby(["AR_ID", "station_id", "role", "series", "network_distance_km", "period"],
                           dropna=False)["anomaly"].mean().unstack("period")
              .reindex(columns=["pre", "post"]))
    change = change.dropna(subset=["pre", "post"])
    change["change"] = change["post"] - change["pre"]
    return change.reset_index()[["AR_ID", "station_id", "role", "series", "network_distance_km", "change"]], days


def dam_summary(changes):
    """Per series: treated, control and matched BACI estimates over dams."""
    columns = ["series", "estimate", "dams", "mean_change", "ci_low", "ci_high", "pct_dams_increased"]
    if changes.empty:
        return pd.DataFrame(columns=columns), None
    dams = changes.groupby(["series", "AR_ID", "role"])["change"].mean().unstack("role")
    for role in ["treated", "control"]:
        if role not in dams:
            dams[role] = np.nan
    dams["baci"] = dams["treated"] - dams["control"]
    rows = []
    for series, d in dams.groupby(level="series"):
        for estimate in ["treated", "control", "baci"]:
            mean, low, high = dam_bootstrap(d[estimate])
            values = d[estimate].dropna()
            rows.append({"series": series, "estimate": estimate, "dams": len(values),
                         "mean_change": mean, "ci_low": low, "ci_high": high,
                         "pct_dams_increased": round(100 * (values > 0).mean()) if len(values) else np.nan})
    return pd.DataFrame(rows, columns=columns), dams


changes, days = pair_changes(df)
results, dam_changes = dam_summary(changes)
display(results.pivot(index="series", columns="estimate",
                      values=["mean_change", "ci_low", "ci_high", "dams", "pct_dams_increased"])
        .reindex(SERIES_OF_INTEREST).round(3))

In [ ]:
# Treated, control and BACI per series. The BACI interval crossing 0 means no evidence
# that the downstream change differs from the upstream one at the same dams.
fig, axes = facet_axes(len(SERIES_OF_INTEREST), width=3.6, height=2.6)
order = ["treated", "control", "baci"]
for ax, series in zip(axes, SERIES_OF_INTEREST):
    r = results[results["series"] == series].set_index("estimate").reindex(order)
    ok = r["mean_change"].notna()
    ax.errorbar(r.loc[ok, "mean_change"], np.arange(3)[ok.to_numpy()],
                xerr=[(r["mean_change"] - r["ci_low"])[ok], (r["ci_high"] - r["mean_change"])[ok]],
                fmt="o", capsize=3)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(range(3))
    ax.set_yticklabels([f"{e} ({int(r.loc[e, 'dams']) if pd.notna(r.loc[e, 'dams']) else 0} dams)"
                        for e in order], fontsize=8)
    ax.set(title=series, xlabel=f"change ({y_label(series)})")
fig.tight_layout()
plt.show()

## 3. Measurement consistency

If the provider, the reported unit or the method changes across the removal, or detection limits fall, the data can show a change the water didn't have. Each check below re-estimates the **treated** dam-level change on pairs measured the same way throughout (or, for censoring, without censored values). Estimates that stay close to "all pairs" mean that source of bias doesn't matter; a large shift means it does and that series needs the restriction.

In [ ]:
def same_throughout(column):
    """Rows of dam-station-series pairs whose `column` has one value (nulls count as one) pre and post."""
    key = ["AR_ID", "station_id", "series"]
    n = df.assign(v=df[column].astype("string").fillna("<null>")).groupby(key)["v"].transform("nunique")
    return df[n == 1]


subsets = {
    "all pairs": df,
    "one provider": same_throughout("provider"),
    "one reported unit": same_throughout("result_unit_raw"),
    "one method": same_throughout("method_id"),
    "uncensored values only": df[~df["is_censored"]],
}
consistency = []
for label, subset in subsets.items():
    est, _ = dam_summary(pair_changes(subset)[0])
    consistency.append(est[est["estimate"] == "treated"].assign(subset=label))
consistency = pd.concat(consistency, ignore_index=True)

fig, axes = facet_axes(len(SERIES_OF_INTEREST), width=3.6, height=2.6)
labels = list(subsets)
for ax, series in zip(axes, SERIES_OF_INTEREST):
    r = consistency[consistency["series"] == series].set_index("subset").reindex(labels)
    ok = r["mean_change"].notna()
    ax.errorbar(r.loc[ok, "mean_change"], np.arange(len(labels))[ok.to_numpy()],
                xerr=[(r["mean_change"] - r["ci_low"])[ok], (r["ci_high"] - r["mean_change"])[ok]],
                fmt="o", capsize=3)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels([f"{l} ({int(r.loc[l, 'dams']) if pd.notna(r.loc[l, 'dams']) else 0})"
                        for l in labels], fontsize=8)
    ax.set(title=f"{series}: treated change", xlabel=y_label(series))
fig.tight_layout()
plt.show()

In [ ]:
# Turbidity: reported units before vs after (NTU, FNU and NTRU have no conversion; spec 3.6.4).
turbidity = df[df["parameter_std"] == "turbidity"]
if len(turbidity):
    display(pd.crosstab(turbidity["result_unit_raw"].fillna("<null>"), turbidity["period"],
                        normalize="columns").mul(100).round(1))

# Detection limits across the removal (nutrients). A censored value is half its detection
# limit (spec 3.6.3), so 2 x value is the limit in the standard unit. A falling limit
# makes concentrations look like they fell.
censored = df[df["is_censored"] & df["parameter_std"].isin(["phosphorus", "orthophosphate", "nitrate", "ammonia"])]
if len(censored):
    limits = (censored.assign(limit=2 * censored["result_value_std"])
              .groupby(["series", "years_from_removal"])["limit"].median().unstack())
    share = (df[df["parameter_std"].isin(["phosphorus", "orthophosphate", "nitrate", "ammonia"])]
             .groupby(["series", "years_from_removal"])["is_censored"].mean().mul(100).round(1).unstack())
    print("Median detection limit by year from removal (standard unit):")
    display(limits.round(4))
    print("% of values censored by year from removal:")
    display(share)

## 4. Event study by dam

A pooled mean per year from removal mixes different dams in different years. Here each dam's yearly mean anomaly (treated stations) is computed first:
- thin gray lines: individual dams in the balanced panel (data in every year of `BALANCED_YEARS`);
- blue: equal-weight mean over all dams with data that year;
- orange: equal-weight mean over the balanced panel only, so the same dams make every point;
- numbers: dams behind each blue point.

A removal effect looks like a step at year 0 in both means. A smooth slope through year 0 is a trend that was there before the removal.

In [ ]:
treated_days = days[days["role"] == "treated"]
dam_year = (treated_days.groupby(["series", "AR_ID", "years_from_removal"])["anomaly"].mean()
            .rename("anomaly").reset_index())

fig, axes = facet_axes(len(SERIES_OF_INTEREST))
for ax, series in zip(axes, SERIES_OF_INTEREST):
    d = dam_year[dam_year["series"] == series]
    if d.empty:
        continue
    wide = d.pivot(index="AR_ID", columns="years_from_removal", values="anomaly")
    balanced = wide.dropna(subset=[y for y in BALANCED_YEARS if y in wide.columns]) \
        if set(BALANCED_YEARS) <= set(wide.columns) else wide.iloc[0:0]
    for _, row in balanced.head(200).iterrows():
        ax.plot(row.index, row.values, color="gray", alpha=0.15, lw=0.7)
    pooled = d.groupby("years_from_removal")["anomaly"].agg(["mean", "size"])
    ax.plot(pooled.index, pooled["mean"], "o-", color="tab:blue", label="all dams")
    if len(balanced):
        ax.plot(balanced.columns, balanced.mean(), "s--", color="tab:orange",
                label=f"balanced ({len(balanced)} dams)")
    low = np.nanpercentile(d["anomaly"], 5)
    for year, n in pooled["size"].items():
        ax.annotate(str(n), (year, low), fontsize=7, ha="center", color="tab:blue")
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_ylim(np.nanpercentile(d["anomaly"], 2), np.nanpercentile(d["anomaly"], 98))
    ax.set(title=series, xlabel="years from removal", ylabel=f"anomaly ({y_label(series)})")
    ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## 5. Distance within dams

Comparing near and far stations across *different* dams mixes distance with watershed. Here only dams with at least two treated stations for a series count: their nearest and farthest stations are compared. If removal effects are local, the nearest station should change more: |near| − |far| > 0. With the 10 km link limit, "far" is at most 10 km.

In [ ]:
treated = changes[changes["role"] == "treated"].dropna(subset=["network_distance_km"])
rows = []
for (series, dam), g in treated.groupby(["series", "AR_ID"]):
    if g["station_id"].nunique() < 2 or g["network_distance_km"].nunique() < 2:
        continue
    near = g.loc[g["network_distance_km"].idxmin()]
    far = g.loc[g["network_distance_km"].idxmax()]
    rows.append({"series": series, "AR_ID": dam, "near_km": near["network_distance_km"],
                 "far_km": far["network_distance_km"], "near_minus_far": near["change"] - far["change"],
                 "abs_near_minus_abs_far": abs(near["change"]) - abs(far["change"])})
within = pd.DataFrame(rows, columns=["series", "AR_ID", "near_km", "far_km", "near_minus_far",
                                     "abs_near_minus_abs_far"])
table = []
for series, g in within.groupby("series"):
    for column in ["near_minus_far", "abs_near_minus_abs_far"]:
        mean, low, high = dam_bootstrap(g[column])
        table.append({"series": series, "measure": column, "dams": len(g), "mean": mean,
                      "ci_low": low, "ci_high": high,
                      "median_near_km": g["near_km"].median(), "median_far_km": g["far_km"].median()})
pd.DataFrame(table).round(3)

## How to read this notebook

- **Section 2 is the headline.** A BACI interval that excludes 0 *and* holds up in section 3 is the strongest evidence these data can give of a removal effect. Treated and control moving together is regional change, not the removal.
- **Section 3** says which series need a measurement restriction before modeling (e.g. turbidity to one unit, nutrients without censored values).
- **Section 4**: compare the blue and orange lines. If they differ, the pooled curve in `analysis_eda` was shaped by which dams had data when, not by the removals.
- **Section 5** tests the expectation that effects are strongest near the dam, without the between-watershed confound.

Limits: the anomaly removes level and season, not flow; upstream controls are few for some series; only removed dams are known, so a standing dam between a station and the removal is invisible.